In [34]:
import numpy as np
%load_ext autoreload
%autoreload 2
from ctd_toolkit.grid import *
from ctd_toolkit.functional_principal_component import *
import matplotlib.pyplot as plt
plt.rcParams.update({"text.usetex": True, "font.family": "serif", "font.serif": ["Computer Modern"]})
%matplotlib qt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Traceback (most recent call last):
  File "C:\Users\m1_gui01\Desktop\CTD_Toolkit\.venv\Lib\site-packages\matplotlib\backends\backend_qt.py", line 523, in _draw_idle
    self.draw()
  File "C:\Users\m1_gui01\Desktop\CTD_Toolkit\.venv\Lib\site-packages\matplotlib\backends\backend_agg.py", line 382, in draw
    self.figure.draw(self.renderer)
  File "C:\Users\m1_gui01\Desktop\CTD_Toolkit\.venv\Lib\site-packages\matplotlib\artist.py", line 94, in draw_wrapper
    result = draw(artist, renderer, *args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\m1_gui01\Desktop\CTD_Toolkit\.venv\Lib\site-packages\matplotlib\artist.py", line 71, in draw_wrapper
    return draw(artist, renderer)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\m1_gui01\Desktop\CTD_Toolkit\.venv\Lib\site-packages\matplotlib\figure.py", line 3257, in draw
    mimage._draw_list_compositing_images(
  File "C:\Users\m1_gui01\Desktop\CTD_Toolkit\.venv\Lib\site-packages\matplotlib\image.py", li

In [ ]:
path = 'C:/Users/m1_gui01/Desktop/postdoc/oceano/profile_list.parquet'
depth_range = DepthRange(0, 1000, 50)
time_range = TemporalRange("2004-01-01", "2024-12-31", "MS")
lat_range = SpatialRange(-75, -40, 0.25)
lon_range = SpatialRange(30, 120, 0.25)
grid = SpatioTemporalGrid(
    latitude=lat_range,
    longitude=lon_range,
    depth = depth_range,
    time=time_range,
    crs="EPSG:4326")

In [ ]:
inst =  FunctionalPrincipalComponent(grid, path = path, source = SQLQuery(), var = ['PSAL_ADJUSTED', 'TEMP_ADJUSTED'])
inst.grid.build()

In [ ]:
inst.depth_min = 5
inst.depth_max = 500
inst.depth_reconstruction = 200

#### LOAD IN SITU

In [ ]:
# IN SITU
df = inst.corresponding_files()

In [ ]:
inst.get_profiles(df)

In [ ]:
inst.save_raw('C:/Users/m1_gui01/Desktop/postdoc/oceano/raw_data')

In [28]:
inst.load_raw('C:/Users/m1_gui01/Desktop/postdoc/oceano/raw_data')

In [ ]:
inst.year_distribution(save = True)
inst.map_distribution(save = True)
inst.month_distribution(save = True)
inst.depth_distribution(save = True)

In [29]:
inst.filter_depth()

C:\Users\m1_gui01\Desktop\CTD_Toolkit\src\ctd_toolkit\functional_principal_component.py:18: RuntimeWarning: All-NaN slice encountered
  from scipy.linalg import cholesky, solve_triangular
C:\Users\m1_gui01\Desktop\CTD_Toolkit\src\ctd_toolkit\functional_principal_component.py:19: RuntimeWarning: All-NaN slice encountered
  from scipy.linalg import cho_factor, cho_solve


In [ ]:
inst.year_distribution(save = True)
inst.map_distribution(save = True)
inst.month_distribution(save = True)

#### LOAD MODEL

In [ ]:
# MODEL
inst.load_model(path = 'C:/Users/m1_gui01/Desktop/cmems_mod_glo_phy_my_0.083deg_P1M-m_thetao-so_30.00E-120.00E_73.00S-40.00S_0.49-541.09m_2008-01-01-2024-12-01.nc')

In [ ]:
inst.format_model()

### LAUNCH FCPA

In [30]:
inst.base_projection()

In [31]:
if len(inst.var) == 1 :
    inst.fpca_univariate()
else :
    inst.fpca_multivariate()

In [ ]:
inst.visualize_projection(var = 'TEMP_ADJUSTED')

In [58]:
recon = inst.truncated_reconstruction()

100%|█████████▉| 286201/286585 [3:20:34<00:16, 23.06it/s]  

In [ ]:
recon = inst.parallel_reconstruction()

In [ ]:
inst.year_distribution(save = False)
inst.map_distribution(save = False)
inst.month_distribution(save = False)

In [59]:
inst.save_data(save_path = 'C:/Users/m1_gui01/Desktop/')

C:\Users\m1_gui01\Desktop\CTD_Toolkit\src\ctd_toolkit\functional_principal_component.py:565: RuntimeWarning: invalid value encountered in divide
  mean_array = sum_array / count_array


In [ ]:
ref = inst.data['PSAL_ADJUSTED'][9]
depth_target = np.arange(inst.depth_min, inst.depth_max + 1, 1).astype(int)
depth_range = np.arange(int(np.floor(np.nanmin(inst.depth[9]))), np.nanmax(inst.depth[9]) + 1)
_data = np.asarray(ref).flatten()
_data_interp = np.interp(depth_range, inst.depth[9], _data)
_data_interp = _data_interp[np.isin(depth_range, depth_target)]

In [ ]:
inst.depth[9]

In [ ]:
plt.plot(inst.depth[9], ref)
plt.plot(depth_target, _data_interp)

In [ ]:

grid_points = np.arange(inst.depth_min, inst.depth_max + 1)
X_0 = inst.ydf[0].coefficients
X_1 = inst.ydf[1].coefficients
X = np.hstack((X_0, X_1))
phi = np.squeeze(inst.basis(grid_points), axis=2)
y_eval_0 = X_0 @ phi
y_eval_1 = X_1 @ phi
estimsig0 = 1 / inst.data.shape[1] * np.sum((y_eval_0 - inst.data[0]) ** 2, axis=0)
estimsig1 = 1 / inst.data.shape[1] * np.sum((y_eval_1 - inst.data[1]) ** 2, axis=0)
regularization = L2Regularization(LinearDifferentialOperator(2), regularization_parameter=1000)
smoother = BasisSmoother(basis=inst.basis, regularization=regularization, return_basis=True)
sig0 = smoother.fit_transform(FDataGrid(data_matrix=estimsig0, grid_points=grid_points))
sig1 = smoother.fit_transform(FDataGrid(data_matrix=estimsig1, grid_points=grid_points))
estimsig0 = np.squeeze(sig0.coefficients @ phi)
estimsig1 = np.squeeze(sig1.coefficients @ phi)
for i in range(5, 10):
    # create phi for varying depth
    # grid_recon = np.arange(self.depth_min, np.nanmax(self.depth_recon[i]) + 1)
    phi_recon = np.squeeze(inst.basis(inst.depth_recon[i]))
    # Create reconstruction informed based on gps localization
    weights = multivariate_normal.pdf(inst.gps, mean=inst.gps_recon[i], cov=np.array([[5, 0], [0, 5]]))
    norm_weights = weights / np.sum(weights)
    alpha_chap = np.sum(X * norm_weights[:, None], axis=0)
    C = X - alpha_chap
    V = (norm_weights[:, None] * C).T @ C
    sigma2_0 = np.sum(np.diag(V[:inst.K, :inst.K] @ inst.mfpca['W_i']))
    sigma2_1 = np.sum(np.diag(V[inst.K:, inst.K:] @ inst.mfpca['W_i']))
    M = np.diag(np.concatenate([np.repeat(1 / sigma2_0, inst.K), np.repeat(1 / sigma2_1, inst.K)]))
    Mdem = np.sqrt(M)  # M^(1/2)
    Mdeminv = np.linalg.inv(Mdem)
    VWM = Mdem @ inst.mfpca['Wdem'] @ V @ inst.mfpca['Wdem'].T @ Mdem
    values, vectors = np.linalg.eig(VWM)
    vectors = Mdeminv @ inst.mfpca['Wdeminv'] @ vectors
    y0 = inst.data_recon[inst.var[0]][i]
    y1 = inst.data_recon[inst.var[1]][i]
    G_00 = phi_recon.T @ V[:inst.K, :inst.K] @ phi_recon  # inst.mat_cov[:inst.K, :inst.K] @ phi_recon
    G_01 = phi_recon.T @ V[:inst.K, inst.K:] @ phi_recon
    G_10 = phi_recon.T @ V[inst.K:, :inst.K] @ phi_recon
    G_11 = phi_recon.T @ V[inst.K:, inst.K:] @ phi_recon
    mu_0 = phi_recon.T @ alpha_chap[:inst.K]  # @ phi_recon # mu_0 = C[:, :inst.K] @ phi_recon
    mu_1 = phi_recon.T @ alpha_chap[inst.K:]  # @ phi_recon  # mu_1 = C[:, inst.K:] @ phi_recon
    mu = np.concatenate((mu_0, mu_1))  # mu = np.concatenate((mu_0, mu_1))
    condi = np.diag(np.concatenate((estimsig0[:np.nanmax(inst.depth_recon[i]) - inst.depth_min + 1],
                                    estimsig1[:np.nanmax(inst.depth_recon[i]) - inst.depth_min + 1])))
    sigmaY = np.block([[G_00, G_01], [G_10, G_11]]) + condi
    Y = np.concatenate((y0, y1))
    eps0 = vectors.T[:2 * inst.K, :inst.K] @ phi_recon
    eps1 = vectors.T[:2 * inst.K, inst.K:] @ phi_recon
    eps = np.hstack([eps0, eps1])
    R = cholesky(sigmaY)
    pc = values[:2 * inst.K] * (eps @ np.linalg.solve(R, np.linalg.solve(R.T, Y - mu)))
    coefs0 = alpha_chap[:inst.K] + np.sum(vectors[:inst.K] @ np.diag(pc), axis=1)
    coefs1 = alpha_chap[inst.K:] + np.sum(vectors[inst.K:] @ np.diag(pc), axis=1)  # C[i, inst.K:]
    recon = np.vstack((coefs0 @ phi, coefs1 @ phi))
    _data_recon = np.concatenate([_data_recon, recon[:, None, :]], axis=1)
    _timestamp_recon = np.concatenate([_timestamp_recon, [self.timestamp_recon[i]]])
    _gps_recon = np.vstack([_gps_recon, self.gps_recon[i]])
    _source_recon = np.concatenate([_source_recon, [self.source_recon[i]]])
    _fns_recon = np.concatenate([_fns_recon, [self.fns_recon[i]]])
    _profile_recon = np.concatenate([_profile_recon, [self.profile_recon[i]]])

In [ ]:
inst.data.shape, y_eval_0.shape

In [ ]:
np.vstack((recon0, recon1))

In [ ]:
grid_points = np.arange(inst.depth_min, inst.depth_max + 1)
X_0 = inst.ydf[0].coefficients
X_1 = inst.ydf[1].coefficients
X = np.hstack((X_0, X_1))
phi = np.squeeze(inst.basis(grid_points), axis=2)
y_eval_0 = X_0 @ phi
y_eval_1 =X_1 @ phi
estimsig0 = 1/inst.data.shape[1] * np.sum((y_eval_0 - inst.data[0])**2, axis = 0)
estimsig1 = 1/inst.data.shape[1] * np.sum((y_eval_1 - inst.data[1])**2, axis = 0)
regularization = L2Regularization(
    LinearDifferentialOperator(2),
    regularization_parameter=1000)
smoother = BasisSmoother(
    basis=inst.basis,
    regularization=regularization,
    return_basis=True)
sig0 = smoother.fit_transform(FDataGrid(data_matrix=estimsig0, grid_points= grid_points))
sig1 = smoother.fit_transform(FDataGrid(data_matrix=estimsig1, grid_points= grid_points))
estimsig0 = np.squeeze(sig0.coefficients @ phi)
estimsig1 = np.squeeze(sig1.coefficients @ phi)


In [ ]:
inst.gps.shape

In [ ]:
C.shape, phi_recon.shape, inst.mfpca['vectors'].shape, inst.data_recon.shape, inst.data.shape

In [ ]:
eps.shape

In [ ]:
from scipy.stats import multivariate_normal
from numpy.linalg import eig, inv
h = 5
grid_recon = np.arange(inst.depth_min, inst.depth_reconstruction + 1)
phi_recon = np.squeeze(inst.basis(grid_recon))
W_i = inst.basis.gram_matrix()  # (K, K)
W_i = (W_i + W_i.T) / 2
W = np.zeros((inst.K * len(inst.var), inst.K * len(inst.var)))
for i in range(len(inst.var)):
    i0, i1 = i * inst.K, (i + 1) * inst.K
    W[i0:i1, i0:i1] = W_i
W = (W + W.T) / 2  # Ensure symmetry
Wdem = cholesky(W)
Wdeminv = solve_triangular(Wdem, np.eye(Wdem.shape[0]))
for i in range(4, 5) :
    # Create reconstruction informed based on gps localization
    weights = multivariate_normal.pdf(inst.gps, mean=inst.gps_recon[i], cov=np.array([[h, 0],[0, h]]))
    norm_weights = weights / np.sum(weights)
    alpha_chap = np.sum(X * norm_weights[:, None], axis=0)
    C = X - alpha_chap
    V = (norm_weights[:, None] * C).T @ C
    sigma2_0 = np.sum(np.diag(V[:inst.K, :inst.K] @ W_i))
    sigma2_1 = np.sum(np.diag(V[inst.K:, inst.K:] @ W_i))
    M = np.diag(np.concatenate([np.repeat(1 / sigma2_0, inst.K), np.repeat(1 / sigma2_1, inst.K)]))
    Mdem = np.sqrt(M)          # M^(1/2)
    Mdeminv = inv(Mdem)
    VWM = Mdem @ Wdem @ V @ Wdem.T @ Mdem
    values, vectors = eig(VWM)
    mfpca = {}
    mfpca["values"] = values
    mfpca["vectnotWM"] = vectors.copy()
    mfpca["vectors"] = Mdeminv @ Wdeminv @ mfpca["vectnotWM"]
    axe = vectors * np.sqrt(values)
    mfpca["pc"] = C @ W @ M @ mfpca["vectors"]
    mfpca["pval"] = np.round(mfpca["values"] / np.sum(mfpca["values"]) * 100,3)
    y0 = inst.data_recon[0, i]
    y1 = inst.data_recon[1, i]
    G_00 = phi_recon.T @ V[:inst.K, :inst.K] @ phi_recon # inst.mat_cov[:inst.K, :inst.K] @ phi_recon
    G_01 = phi_recon.T @ V[:inst.K, inst.K:] @ phi_recon
    G_10 = phi_recon.T @ V[inst.K:, :inst.K] @ phi_recon
    G_11 = phi_recon.T @ V[inst.K:, inst.K:] @ phi_recon
    mu_0 = phi_recon.T @ alpha_chap[:inst.K] #@ phi_recon # mu_0 = C[:, :inst.K] @ phi_recon
    mu_1 = phi_recon.T @ alpha_chap[inst.K:] #@ phi_recon  # mu_1 = C[:, inst.K:] @ phi_recon
    mu = np.concatenate((mu_0, mu_1)) #mu = np.concatenate((mu_0, mu_1))
    condi = np.diag(np.concatenate((estimsig0[:inst.depth_reconstruction-inst.depth_min+1], estimsig1[:inst.depth_reconstruction-inst.depth_min+1])))
    sigmaY = np.block([[G_00, G_01], [G_10, G_11]]) + condi
    Y = np.concatenate((y0, y1))
    eps0 = mfpca['vectors'].T[:2*inst.K, :inst.K] @ phi_recon
    eps1 = mfpca['vectors'].T[:2*inst.K, inst.K:] @ phi_recon
    eps = np.hstack([eps0, eps1])
    #pc = (inst.mfpca['values'][:2*npc] * eps.T).T @ np.linalg.inv(sigmaY) @ (Y - mu)
    R = cholesky(sigmaY)
    pc = mfpca['values'][:2*inst.K] * (eps @ np.linalg.solve(R, np.linalg.solve(R.T, Y - mu)))
    coefs0 = alpha_chap[:inst.K] + np.sum(mfpca['vectors'][:inst.K] @ np.diag(pc), axis = 1)
    coefs1 = alpha_chap[inst.K:] + np.sum(mfpca['vectors'][inst.K:] @ np.diag(pc), axis = 1) # C[i, inst.K:]
    recon0 = coefs0 @ phi
    recon1 = coefs1 @ phi

In [ ]:
plt.plot(inst.data_ref[0, 4])
plt.plot(recon0)

In [ ]:
2*len(inst.depth_recon)*npc, 195*2

In [ ]:



phi_block = np.block([
    [phi, np.zeros_like(phi)],
    [np.zeros_like(phi), phi]])
mu = phi.T @ inst.alpha.reshape(2, inst.K, -1).T
eps = phi.T @ inst.mfpca['vectors'].reshape(2, inst.K, -1).T
sigmaY = phi_block @ inst.mat_cov @ phi_block.T + np.diag(np.ones(2*inst.K) * 1e-6)
R = cholesky(sigmaY)
values = inst.mfpca['values'][:2*inst.K]
residual = inst.data - mu
tmp = solve_triangular(R.T, residual, lower=False)
tmp2 = solve_triangular(R, tmp, lower=False)
pc = values * (eps.T @ tmp2)
coef = inst.mfpca['alpha_mean'] + inst.mfpca['vectors'] @ pc
inst.reconstructed_data = phi @ coef

In [ ]:
inst.truncated_reconstruction()